# 16. 저장한 표본을 DB에서 다시 읽기

2026-10-02 사용자 승인 후 개인 스키마에 PDF 정보 1개·처리 설정 1개·원문 4개·청크 11개·이미지 3개·연결 3개를 저장했습니다.
COMMIT 후 새 읽기 전용 연결로 값·벡터를 대조했고 처리 상태 ready를 확인했습니다. 테이블·기본 색인 합계는 512KiB입니다.
[수업 개념] 트랜잭션은 여러 저장을 모두 성공하거나 모두 되돌리는 한 묶음으로 다룹니다.
[프로젝트 추가] 모델을 다시 실행하지 않고 이미 저장된 자료를 조회합니다. 이 노트북은 SELECT만 사용하며 저장·업로드 명령은 없습니다.

In [ ]:
# 현재 폴더와 상위 폴더에서 개인 코드를 찾아 src를 등록합니다.
from pathlib import Path
import sys
current = Path.cwd().resolve()
project_folder = next((p for p in [current, *current.parents]
                       if (p / 'src/car_search_rag/zzong_santafe_lag/sample_store.py').is_file()), None)
if project_folder is None:
    raise FileNotFoundError('프로젝트에서 실행하세요.')
if str(project_folder / 'src') not in sys.path:
    sys.path.insert(0, str(project_folder / 'src'))


In [ ]:
# UUID는 저장된 처리 작업의 고유 번호입니다. 비밀번호가 아닙니다.
from uuid import UUID
from car_search_rag.zzong_santafe_lag.sample_store import read_sample_report
run_id = UUID('dc327282-8cfd-4eb3-a2b0-1433e0b088ce')
stored = read_sample_report(run_id)
print('처리 상태:', stored['run_status'])
print('저장 건수:', stored['row_counts'])
print('벡터 크기:', stored['vector_shape'])
print('숫자 유효성:', stored['finite'], '| 정규화:', stored['normalized'])
print('테이블과 색인 용량:', stored['total_table_bytes'] / 1024, 'KiB')


In [ ]:
# DB에서 읽은 원문 기록의 검색용 글과 각주 연결을 확인합니다.
for row in stored['parents']:
    print('\n주제:', row['title'], '| PDF:', row['source_pages'])
    print('함께 읽을 기록:', row['required_context_record_ids'])
    print(row['content'])


In [ ]:
# DB에는 그림 파일 자체가 아니라 현재 PC의 상대 경로와 설명을 저장했습니다.
for link in stored['image_links']:
    print('\n원문:', link['parent_record_id'], '| PDF:', link['pdf_page_number'])
    print('그림 키:', link['pdf_image_key'], '| 경로:', link['local_path'])
    print('그림 설명:', link['description'])
    print('현재 PC에 파일이 있나요?:', (project_folder / link['local_path']).is_file())


## DBeaver에서 직접 보기

Schemas → zzong_santafe_lag → Tables → parent_records를 열고 Data에서 새로 고침하면 4개 기록이 보입니다. chunks는 11개, images와 record_images는 각각 3개입니다.
db/03_sample_readback.sql.txt의 SELECT를 한 문장씩 실행하면 같은 처리 작업만 읽을 수 있습니다.
ready는 저장 확인 완료를 뜻하며 완성된 챗봇이나 전체 PDF 검토 완료를 뜻하지 않습니다.
Storage 업로드, 전체 기록 저장과 DB 자료를 이용한 기존 결합 검색 재구성은 이후 단계입니다.